# EQO notebook: FTQC → IQM preparation

This notebook submits an admitted FTQC *preparation* workflow. FTQC stays in its OCI runtime; no IQM token or hardware credential is supplied from notebook state. Start EQO Local before running it.

In [ ]:
import os

from eqo import EQOClient, render_artifact, render_run

EQO_ENDPOINT = os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080")
eqo = EQOClient.connect(EQO_ENDPOINT)
eqo.health()

In [ ]:
workflow = next(
    (item for item in eqo.workflows.list() if item["id"] == "ftqc-iqm-steane-preparation"),
    None,
)
if workflow is None:
    raise RuntimeError("The FTQC IQM preparation workflow is not published by this EQO profile.")
workflow

## Prepare one logical qubit

Create the typed OpenQASM input artifact. This example prepares one Steane logical qubit; it does **not** route or submit to IQM hardware. For hardware execution, use the separately admitted worker and its worker-local credential flow—not this notebook.

In [ ]:
logical_zero = """OPENQASM 3.0;
include \"stdgates.inc\";

qubit[1] q;
bit[1] result;
result[0] = measure q[0];
"""

input_circuit = eqo.artifacts.create_input(
    "qhpc.quantum-circuit@1",
    logical_zero,
    name="logical-zero.qasm",
    labels={"example": "ftqc-iqm-steane-preparation"},
)
render_artifact(input_circuit)

In [ ]:
run = eqo.workflows.submit(
    workflow["id"],
    workflow["version"],
    inputs={"circuit": input_circuit.id},
)
render_run(run)

In [ ]:
completed = run.wait(timeout=300)
if completed.state != "succeeded":
    raise RuntimeError(f"FTQC preparation ended in {completed.state}; inspect render_run(completed).")

iqm_circuit = completed.artifacts.by_type("qhpc.iqm-circuit@1")
preparation_report = completed.artifacts.by_type("qhpc.ftqc-iqm-preparation-report@1")
render_run(completed), render_artifact(iqm_circuit), render_artifact(preparation_report)